##Agenda

### 3 Pyspark Tips  
     1-   Methods for dataframes(read, format, etc, )
     2-   Select() vs selectExpr() vs expr()
          -WARNING: col() cannot be nested inside expr()
          -Renaming a Column (Alias) vs AS
              Use alias() with "select(col().alias())" 
              Use  'AS'   with "selectExpr()"
          -WARNING 'where' SQL keyword CANNOT be used inside selectExpr()

     3-   Grouping vs agg()
          3.1 .agg(max("column")).first()[0] # max_value
     4-   where/filter/having
     5-   regexp_replace(), replace()
     6-   cast(), try_cast()
     7-   Dates (datediff, interval)







![image_1774985632190.png](./image_1774985632190.png "image_1774985632190.png")




-----------------------------------------------------------
#### 1 Methods for dataframes(read, format, etc)
-----------------------------------------------------------
      spark.read.  Returns a DataFrameReader  to read data in a DataFrame. 
      https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.SparkSession.read.html

      spark.read.format Specifies the file type (can be "json", "parquet", "jdbc", "orc", "text", etc.)..
      https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameReader.format.html#

      spark.read.option  Adds an input option for the underlying data source.
      https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameReader.option.html

      spark.read.load   Loads the data from the specified path(s)
      https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameReader.load.html




-----------------------------------------------------------
#### 2 select() vs selectExpr()

###### Both methods return a new DataFrame and are used for selecting and transforming columns
-----------------------------------------------------------


#####< The core difference is that>

#### select( *cols)
* select() uses PySpark's Column API with expressions(column objects and functions) or raw column name strings. 
###### DataFrame.select( *cols) 
      Where Parameters: cols: str, Column, or list
             column names (string) or expressions (Column). 
             If one of the column names is ‘ * ’, that column is expanded to include all columns in the current DataFrame.

       
#### selectExpr() 
* Accepts standard SQL expressions as strings. 
         but CAREFUL you CANNOT include 'where' statement inside it or do any groupings for a particular dimension
         UNLESS you create a TemporaryView first to execute a whole SQL expresion

-----------------------------------------------------------------------------------------

select() Key Characteristics: 

  -  API: Uses Column objects and Column expressions (col("...")).
  -  Use Case: Ideal for selecting specific columns and basic manipulation.
  -  Example: Select column  objetcs --->      df.select(col("name") ) 
  -  Example: select columns  names  --->      df.select("name")
  -  Example: select df[]     names  --->      df.select(df["name"])
  -  Example: select df.colum names  --->      df.select(df.name)  

selectExpr() Key Characteristics: 

  -  API: Accepts only SQL expressions formatted as strings.
  -  Use Case: Ideal for rapid SQL-style transformations and alias naming.
  -  Example: df.selectExpr("name", "age + 10 as age_plus_10")

Key Differences:

  -  Syntax: select() uses Pythonic column expressions; selectExpr() uses pure string SQL.
  -  Flexibility: selectExpr() is more concise for complex SQL transformations (e.g., when, abs, concat) in a single string.
  -  Performance: Generally, both are efficient, but select() is slightly more direct as it avoids parsing SQL strings.
  -  Renaming: selectExpr() makes renaming columns within expressions more concise. 

.



##### <When to Use Which?>

######   Use select() 
    when you are building transformations programmatically (e.g., looping through a list of Column objects) or when you prefer a strictly Pythonic API style.
-------------------------------------------------------------    
######  Use selectExpr() 
    when you want to write concise, SQL-like transformations without importing the functions module for every simple operation. 
    It is also excellent for users already comfortable with SQL syntax
"""
#####   <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<expr() vs. selectExpr()>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>


##### expr()             
                      operates on a single SQL expression and returns a Column object for use within other DataFrame methods, 
##### selectExpr()        
                      Accepts one or more SQL expressions as strings and applies them across the entire DataFrame in a single call,
                      returning a new DataFrame. 

                      <<python>>
                      # Using selectExpr to perform multiple operations at once
                      
                      -  df_transformed = df.selectExpr( "name", "Balance * 1.02 AS Adjusted_Balance",
                                                        "CASE WHEN Balance > 100000 THEN 'High' ELSE 'Low' END AS Balance_Level"
                                                      )


#### Key differences
expr() is a standalone function that evaluates a single SQL expression into a column, whereas 

selectExpr() is a DataFrame method that lets you run multiple SQL expressions to select and transform columns simultaneously

Feature---------expr()---------------------------------------------------selectExpr()-------------

Type------------Function (imported from pyspark.sql.functions) -----Method (called directly on a DataFrame)

Arguments-----Takes exactly one string expression.------------------Takes one or more comma-separated string expressions.

Usage----------Must be nested inside methods like select(), --------Acts as a standalone drop-in replacement for select().
                withColumn(), or filter().

Output---------Returns a Column object.-----------------------------Returns a new DataFrame.

#####  col(col)           
                    - Se utiliza para hacer referencia a una columna de un DataFrame por su nombre, devolviendo un objeto de tipo Column. 
                    Es fundamental para:
                    -  transformaciones
                    -  filtros 
                    -  ordenaciones al convertir cadenas de texto en expresiones de columna. 
                       col("nombre_columna").

#####  column(col)        
                    - Does the sam e as 'col(col)' but 'col(col)' is more preferred. 
#####  lit(col)           
                    - Creates a Column of literal value.
#####  try_cast(dataType) 
                    - A special version of cast that performs the same operation, but returns a NULL value instead of raising 
                      an error if the invoke method throws exception.
"""

In [0]:
"""

The core difference between PySpark's
The core difference is that 
1)    select() -------> accepts column objects or raw column name strings. Uses the DataFrame API's column objects and functions
2)    selectExpr() ---> accepts SQL-style expression strings
3)    expr() ---------> is a function that parses a SQL expression string into a Column object. 
                        Useful when you want to apply a SQL expression to a DataFrame column or perform a SQL-like operation on a DataFrame.

          Example: # 1. Combining Column objects and SQL strings within a standard select()
                      df.select(col("name"), expr("age + 1 as next_year_age"))

          Example: # 2. Using it inside a filter condition
                      df.filter(expr("age > 18 AND status = 'Active'"))

          Example: # 3. Using expr() with sum, max, etc.
             
                        Calculates the total sum of a column
                        df.agg(expr("sum(sales)")).show()

                        Evaluates conditional math inside the sum
                        df.select(expr("sum(price * quantity)")).show() 

          WARNING: col() cannot be nested inside expr() because :
                    expr() expects a raw SQL STRING but returns a Column object, you can use it interchangeably with col() 
                    col() returns a Column object NOT A STRING.
                                  
                    # ❌ THIS WILL FAIL
                      df.select(expr(col("age") + 1))


<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<expr() vs. selectExpr()>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

*expr()              operates on a single SQL expression and returns a Column object for use within other DataFrame methods, 
*selectExpr()        Accepts one or more SQL expressions as strings and applies them across the entire DataFrame in a single call, returning a new
                     DataFrame. 
                     <<python>>
                     # Using selectExpr to perform multiple operations at once
                     df_transformed = df.selectExpr( "name",
                                                     "Balance * 1.02 AS Adjusted_Balance",
                                                      "CASE WHEN Balance > 100000 THEN 'High' ELSE 'Low' END AS Balance_Level"
                                                    )
<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<< .col vs. .column >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
col and column are functionally identical. 
Both are functions within the pyspark.sql.functions module that return a Column object based on a given string name.

Convention: 
col is much more commonly used in the PySpark community because it is shorter and more concise

col(col)           - Se utiliza para hacer referencia a una columna de un DataFrame por su nombre, devolviendo un objeto de tipo Column. 
                     Es fundamental para transformaciones, filtros y ordenaciones al convertir cadenas de texto en expresiones de columna. 
                     col("nombre_columna").
column(col)        - Does the sam e as 'col(col)' but 'col(col)' is more preferred. 


lit(col)           - Creates a Column of literal value.
try_cast(dataType) - A special version of cast that performs the same operation, but returns a NULL value instead of raising an error if 
                     the invoke method throws exception.
"""

'\n\nThe core difference between PySpark\'s\nThe core difference is that \n1)    select() ------->accepts column objects or raw column name strings. Uses the DataFrame API\'s column objects and functions\n2)    selectExpr() --->accepts SQL-style expression strings\n3)    expr() --------->is a function that parses a SQL expression string into a Column object. \n             Useful when you want to apply a SQL expression to a DataFrame column or perform a SQL-like operation on a DataFrame.\n\n          Example: # 1. Combining Column objects and SQL strings within a standard select()\n                      df.select(col("name"), expr("age + 1 as next_year_age"))\n\n          Example: # 2. Using it inside a filter condition\n                      df.filter(expr("age > 18 AND status = \'Active\'"))\n\n          Example: # 3. Using expr() with sum, max, etc.\n             \n                        Calculates the total sum of a column\n                        df.agg(expr("sum(sales)")).show()

----------------------------------------------------------
#### select() WARNINGS

-------------------------------------------------------------
#### selectExpr() WARNINGS
                    YOU CANNOT put groupby inside selectExpr() 
                    ❌ THIS WILL FAIL: raw_emp_df.selectExpr(" name, count(*) groupby(name)"))
                    ✅ THIS WILL WORK: raw_emp_df.selectExpr("name").groupBy("name").count()

----------------------------------------------------------------------------
                   YOU CANNOT put 'where' SQL keyword inside selectExpr()
                     The selectExpr() method is designed for selecting columns and applying SQL expressions to those 
                     columns(e.g., transformations, aggregations, or conditional logic with CASE WHEN), not to filter the dataset  

                     ✅ THIS WILL WORK: raw_emp_df.filter(raw_emp_df.departmentid.isNull())
                                                   .selectExpr('name', 'departmentid')


----------------------------------------------------------
#### expr() WARNINGS 

###### WARNING: col() cannot be nested inside expr()
                    You can HAVE col() and expr() INSIDE  A select() because both return a col object:   df.select(col("name"), expr("age + 1 as next_year_age")) 
                    
                    BUT CANNO PUT col() INSIDE A expr() !!!!!!!!!!!!!!!!!
                    expr() expects a raw SQL STRING but returns a Column object, you can use it interchangeably with col() 
                    col() returns a Column object NOT A STRING.
                                  
                    # ❌ THIS WILL FAIL
                      df.select(expr(col("age") + 1))

--------------------------------------------------------------------
##### Examples Renaming a Column (Alias) vs AS

In [0]:
In select(), you must use a Column object to call .alias(). 
# select()
#from pyspark.sql import functions as F
#df.select(F.col("age").alias("current_age"))

In selectExpr(), you use the SQL 'AS' keyword.
# selectExpr()
#df.selectExpr("age AS current_age")

In expr(), parses a SQL expression string into a Column object, allowing you to write raw SQL logic directly inside DataFrame transformations. 




###### Examples Mathematical Operations & SQL Functions 

selectExpr() Is not designed to filter the dataset 

is often more concise for simple calculations or applying built-in SQL functions like abs(), upper(), or cast().



In [0]:
# select()
#df.select((F.col("age") + 10).alias("age_plus_ten"))

# selectExpr()
#df.selectExpr("age + 10 AS age_plus_ten", "CAST(id AS STRING)")



![image_1774641851028.png](./image_1774641851028.png "image_1774641851028.png")

-----------------------------------------------------------
#### 3 groupBy() and agg()
-----------------------------------------------------------

##### ----> What is GroupBy?

PySpark groupBy( ) is used to split the data into groups based on one or more columns, which can then be aggregated or transformed independently.


Group data
###### 1- grouped = df.groupBy("department")

Once data is grouped, you can  apply aggregations .count() or .sum() to obtain a new DataFrame.

###### 2- grouped.count().show()

##### ----> Parameters and return values

The only parameter for the method is *cols which accepts:
- column names 
- column expressions
- column ordinals (int) 
- list of columns.




------------------------------------------------------------------------------
#### 3.1  .agg(max("column")).first()[0] # max_value
-------------------------------------------------------------------------------

In PySpark, the sequence agg(...).first()[0] is a common pattern used to extract a single scalar value from an aggregated DataFrame.

max_salary = raw_emp_df.agg(max("salary")).first()[0] # max_salary

In [0]:
""" Breakdown of the Sequence:

    raw_emp_df: Es el nombre del DataFrame que contiene los datos de los empleados.
    .agg(max("salary")):
        agg es la función de agregación.
        max_("salary") calcula el valor máximo de la columna llamada "salary".
        Nota: El resultado de este paso sigue siendo un DataFrame con una sola fila y una sola columna.
    .first()[0]:
        Esta es una "acción" de Spark que toma la primera fila del DataFrame resultante y la devuelve como un objeto tipo Row de Spark
        [0]: Accede al primer elemento de esa fila (el valor numérico del salario máximo). 
        Sin esto, tendrías un objeto Row(max(salary)=5000), pero con el [0] obtienes simplemente el 5000.
"""

'\nBreakdown of the Sequence:\n\n    .agg(...): Performs an aggregation on a DataFrame (e.g., sum, max, avg). It returns a new DataFrame with usually just one row (if no groupBy was used) or one row per group.\n    .first(): A pyspark.sql.DataFrame.first action that returns the first row of the resulting DataFrame as a Row object.\n    [0]: Uses index-based access on that Row object to retrieve the value from the first column. \n'

##### DatFrame example data

In [0]:
from pyspark.sql import Row

data = [
    Row(department="Sales", employee="Alice", salary=5000),
    Row(department="Sales", employee="Bob", salary=4800),
    Row(department="HR", employee="Carol", salary=4000),
    Row(department="HR", employee="David", salary=3900),
    Row(department="IT", employee="Eve", salary=6000)
]

df = spark.createDataFrame(data)
df.show()

+----------+--------+------+
|department|employee|salary|
+----------+--------+------+
|     Sales|   Alice|  5000|
|     Sales|     Bob|  4800|
|        HR|   Carol|  4000|
|        HR|   David|  3900|
|        IT|     Eve|  6000|
+----------+--------+------+



In [0]:
#grouping by column NAME like above
df.groupBy("department")

#grouping by column EXPRESSION
df.groupBy(df.department)

#grouping by column ORDINAL
df.groupBy(1)

#grouping by LIST of COLUMNS, you can mix the methods!
df.groupBy(["department", 2])


##### ----> GroupBy on single and multiple columns

In [0]:
#grouping by SINGLE COLUMN
df.groupBy("department").sum(“salary”).show()

#grouping by MULTIPLE COLUMNS
df.groupBy(["department", 2]).sum(“salary”).show()

##### ----> Multiple aggregations with agg()

We do not need to aggregate on the same column with agg(), you can define a different column for each aggregation function.

In [0]:
# After already starting your session
from pyspark.sql.functions import count, sum, avg, max, min

df.groupBy("department").agg( count("employee").alias("employee_count"), 
                              avg("salary").alias("avg_salary"),
                              max("salary").alias("max_salary")
                            ).show()
  

+----------+--------------+----------+----------+
|department|employee_count|avg_salary|max_salary|
+----------+--------------+----------+----------+
|     Sales|             2|    4900.0|      5000|
|        HR|             2|    3950.0|      4000|
|        IT|             1|    6000.0|      6000|
+----------+--------------+----------+----------+



##### ----> Multiple aggregations with agg( expr() )

You can use aggregation and expr() function but apply these changes:

expr() expects a string argument (a SQL expression), but in teh code above the functions count(), avg(), and max() return Column objects, not strings.

The fix is simple: expr() should wrap the SQL expression as a string, not wrap Column objects.

In [0]:
# After already starting your session
from pyspark.sql.functions import count, sum, avg, max, min, col, expr

df.groupBy("department").agg( expr("count(employee) as employee_count"), 
                              expr("avg(salary) as avg_salary"),
                              expr("max(salary) as max_salary")
                              
                            ).show()

+----------+--------------+----------+----------+
|department|employee_count|avg_salary|max_salary|
+----------+--------------+----------+----------+
|     Sales|             2|    4900.0|      5000|
|        HR|             2|    3950.0|      4000|
|        IT|             1|    6000.0|      6000|
+----------+--------------+----------+----------+



##### Advanced aggregations 
https://www.datacamp.com/tutorial/pyspark-groupby
###### ---> Pivoting
###### ---> Rollups and cubes
###### ---> Grouping sets
###### ---> Custom aggregation functions
###### ---> PySpark groupBy Performance Optimization Strategies


##### ----> Filtering Aggregated Data

In PySpark, you can filter groups based on aggregate metrics post-grouping using the filter( ) or where( ) methods. 
-   You need to provide a condition in either Python or SQL expressions. 
-   You can filter BEFORE or AFTER the aggregation. 
    -   Filtering before will impact the aggregation by limiting what data gets aggregated and can improve performance. 



##### ----> PySpark SQL GROUP BY Query using  a temporary view 

If you are more comfortable writing SQL, use the SQL API to write statements.

###### ---> First create a temporary view 
First step is to create a temporary view using the createOrReplaceTempView() method of the DataFrame
Then you can use spark.sql() to write your statement.

In [0]:
# Create a temporary view using the DataFrame
df.createOrReplaceTempView("employees")

# Write a SQL-like statement
sql_query = """
    SELECT department, AVG(salary) AS avg_salary
    FROM employees
    GROUP BY department
"""

result_df =  spark.sql(sql_query)
result_df.display()


---------------------------------------------------------------
#### selectExpr() 
###### In PySpark  You can use selectExpr() to perform aggregations in two primary ways: 

  -     1 Grouped Aggregation (after a groupBy()).  Use groupBy() then use agg() or selectExpr() 
  -     2 Global Aggregation: either on the entire DataFrame 


#### 1. Grouped Aggregation (after groupBy())  <<This is the STANDARD/COMMON way>>
To aggregate data within specific groups, you 
   - first use groupBy() and then
   - use agg() or selectExpr() on the resulting GroupedData object. 

Note that selectExpr() is generally called after the groupBy to select the final columns, which often results in the same effect as using agg()

A common approach to applying aggregation expressions using SQL syntax within the DataFrame API is to use the agg() method on the grouped data:

##### This is teh standard way in pyspark to groupe aggregations. THIS PIECE CONTAINS A field("department") and aggregations(sum and avg) but the field is grouped first

In [0]:

"""from pyspark.sql.functions import col, sum, avg
df.groupBy("department")
  .agg(sum("salary").alias("total_salary"),
       avg("salary").alias("avg_salary")
      )
"""


##### using selectExpr() after groupBy()


In [0]:
"""
from pyspark.sql import functions as F

# Sample Data
data = [("Sales", 100), ("Sales", 200), ("Marketing", 50)]
df = spark.createDataFrame(data, ["department", "revenue"])

# Aggregation followed by selectExpr
result = df.groupBy("department") \
           .agg(F.sum("revenue").alias("total")) \
           .selectExpr("department", "total", "total * 1.1 as revenue_with_tax")

result.show()

"""

'\nfrom pyspark.sql import functions as F\n\n# Sample Data\ndata = [("Sales", 100), ("Sales", 200), ("Marketing", 50)]\ndf = spark.createDataFrame(data, ["department", "revenue"])\n\n# Aggregation followed by selectExpr\nresult = df.groupBy("department")            .agg(F.sum("revenue").alias("total"))            .selectExpr("department", "total", "total * 1.1 as revenue_with_tax")\n\nresult.show()\n\n'

##### If you prefer to use the groupBy() INSIDE THE selectExpr(), you would typically register the DataFrame as a temporary view first, and then run a SQL query:


In [0]:
# Using selectExpr() with groupBy is less direct than using agg()
# A common pattern is to register a temp view and use spark.sql()
#df.createOrReplaceTempView("products_view")
#spark.sql("SELECT Product, AVG(Price) AS avg_price, SUM(Quantity) AS total_quantity FROM products_view GROUP BY Product").show()


-------------------------------------------------------


##### 2. Global Aggregation (on the entire DataFrame) 
You can perform aggregations on the entire DataFrame without explicit grouping. This is a shorthand for df.groupBy().agg()

In [0]:
"""
from pyspark.sql import SparkSession

# Create a SparkSession (if not already created)
spark = SparkSession.builder.appName("selectExprAgg").getOrCreate()

# Sample DataFrame
data = [("Laptop", 1500, 10), 
        ("Mouse", 50, 200),
        ("Laptop", 1200, 5),
        ("Keyboard", 100, 50)]
columns = ["Product", "Price", "Quantity"]
df = spark.createDataFrame(data, columns)
df.show()


# Global aggregation using selectExpr()
df.selectExpr(
    "avg(Price) AS avg_price",
    "sum(Quantity) AS total_quantity",
    "count(Product) AS total_products"
).show()
"""


##### PAY ATTENTION!! This piece of teh above code DOES NOT apply aggregations to a GROUPBY  any DIMENSION/FIELD, IT ONLY DOES AGGREGATIONS



------------------------------------------------------------------------------
#### 3.1  .agg(max("column")).first()[0] # max_value
-------------------------------------------------------------------------------

In PySpark, the sequence agg(...).first()[0] is a common pattern used to extract a single scalar value from an aggregated DataFrame.

max_salary = raw_emp_df.agg(max("salary")).first()[0] # max_salary

In [0]:
"""
Breakdown of the Sequence:

    raw_emp_df: Es el nombre del DataFrame que contiene los datos de los empleados.
    .agg(max("salary")):
        agg es la función de agregación.
        max_("salary") calcula el valor máximo de la columna llamada "salary".
        Nota: El resultado de este paso sigue siendo un DataFrame con una sola fila y una sola columna.
    .first()[0]:
        Esta es una "acción" de Spark que toma la primera fila del DataFrame resultante y la devuelve como un objeto tipo Row de Spark
        [0]: Accede al primer elemento de esa fila (el valor numérico del salario máximo). 
        Sin esto, tendrías un objeto Row(max(salary)=5000), pero con el [0] obtienes simplemente el 5000.
"""

'\nBreakdown of the Sequence:\n\n    .agg(...): Performs an aggregation on a DataFrame (e.g., sum, max, avg). It returns a new DataFrame with usually just one row (if no groupBy was used) or one row per group.\n    .first(): A pyspark.sql.DataFrame.first action that returns the first row of the resulting DataFrame as a Row object.\n    [0]: Uses index-based access on that Row object to retrieve the value from the first column. \n'



-----------------------------------------------------------
#### 4  where/filter/having
-----------------------------------------------------------

##### In PySpark
###### where() and filter() 
Both are used to filter rows based on a given condition, similar to the SQL WHERE clause. They are interchangeable and can accept boolean expressions or SQL-style string conditions

###### having() 
In PySpark, 'having' IS NOT USED. You have two options instead:
  -     Applying a filter() or where() condition after the groupBy() and aggregation steps have been performed
  -     Using SQL queries directly 


### Example: Multiple filters

##### SQL Query

###### What San Francisco neighborhoods in in the zip codes 94102 and 94103



In [0]:
%sql
SELECT City, Neighborhood, Zipcode
FROM dev.spark_db.sf_fire_calls
WHERE City = 'SF' and Zipcode in (94102 , 94103);

#####Pyspark

###### isin(), &

In [0]:
from pyspark.sql.window    import Window
from pyspark.sql.functions import rank, col, count, sum, expr, desc

result_df = ( raw_fire_df.where( (raw_fire_df["Zipcode"].isin([94102 , 94103])) & (raw_fire_df["City"]=='SF') )
                         .select("City", "Neighborhood", "Zipcode")
            )

result_df.display()


-----------------------------------------------------
#### 5 regexp_replace()  
-----------------------------------------------------------

##### To replace strings in PySpark, 
the most efficient methods use built-in functions like:

- regexp_replace for pattern matching or 
- the DataFrame.replace method for direct value substitution.


##### 1. regexp_replace (Pattern-based)
Used for complex string manipulation within a column using regular expressions. It replaces every substring that matches a specific pattern
- Syntax: regexp_replace(column, pattern, replacement)
- Key Feature: Supports regex characters like \d (digits), ^ (start), and $ (end).


In [0]:
"""
from pyspark.sql.functions import regexp_replace
# Replaces all digits with 'X' in the "phone" column
df.withColumn("masked_phone", regexp_replace("phone", r"\d", "X"))

"""

##### 2. replace (Value-based)
There are two distinct "replace" methods in PySpark:

Method 	---------------Level ----------	Description
- df.replace() ----	 DataFrame	--- Replaces exact values across the whole DataFrame or a subset of columns.
- F.replace()	------ Column	   ------ Replaces exact substrings within a string column (Introduced in Spark 3.5.0).


###### DataFrame replace Example:

In [0]:
"""
# Replaces "Male" with "M" in the "Gender" column
df.replace("Male", "M", subset=["Gender"])

# Using a dictionary for multiple replacements at once
df.replace({"John": "Jonathan", "Jane": "Janet"}, subset=["Name"])

"""

###### Column-Level replace Example:

In [0]:
"""
from pyspark.sql import functions as F
# Replaces the literal substring "lane" with "ln"
df.withColumn("address", F.replace("address", "lane", "ln"))

"""

'\nfrom pyspark.sql import functions as F\n# Replaces the literal substring "lane" with "ln"\ndf.withColumn("address", F.replace("address", "lane", "ln"))\n\n'

Summary of Differences

-----------------------------------------
https://www.google.com/search?q=regexp_replace+and+replace+pyspark&client=firefox-b-d&hs=1EyU&sca_esv=91aac8a642e70fa5&biw=1680&bih=739&sxsrf=ANbL-n5PaxFdRNdA4xxVEC9Sgxk7KUL7nw%3A1775176464643&ei=EAvPad-AJ4yrur8P7OKj-Qs&oq=regexp_replace+and+replace+pysp&gs_lp=Egxnd3Mtd2l6LXNlcnAiH3JlZ2V4cF9yZXBsYWNlIGFuZCByZXBsYWNlIHB5c3AqAggAMgUQIRigATIFECEYoAFIuqUBUABYjJEBcAJ4AZABAJgB3gGgAZYTqgEGMS4xNy4xuAEDyAEA-AEBmAIVoAL_FcICCxAuGIAEGLEDGIMBwgIIEC4YgAQYsQPCAggQABiABBixA8ICBRAAGIAEwgIaEC4YgAQYsQMYgwEYlwUY3AQY3gQY4ATYAQHCAgQQIxgnwgIMECMYgAQYExgnGIoFwgIKEAAYgAQYQxiKBcICFhAuGIAEGLEDGNEDGEMYgwEYxwEYigXCAgoQIxiABBgnGIoFwgIKEC4YgAQYQxiKBcICDRAAGIAEGLEDGEMYigXCAgoQABiABBgUGIcCwgIIEAAYgAQYywHCAgcQABiABBgKwgIGEAAYDRgewgIIEAAYChgNGB7CAgQQABgewgIIEAAYFhgKGB7CAgYQABgWGB7CAggQABiABBiiBMICBRAAGO8FmAMAugYGCAEQARgUkgcGMi4xNy4yoAezc7IHBjAuMTcuMrgH8RXCBwswLjMuOC43LjEuMsgHjgKACAA&sclient=gws-wiz-serp







-----------------------------------------------------------
#### 6  cast(), try_cast()
-----------------------------------------------------------

In PySpark,
cast() and try_cast() are used to convert a column from one data type to another, but they handle conversion failures differently, especially when ANSI mode is enabled

Key Differences
Feature 	

cast()	
- Success: Returns the converted value.
- Failure (Non-ANSI):	Usually returns null.	
- Failure (ANSI Mode):	Throws an error (e.g., SparkArithmeticException).

try_cast()
- Success:	Returns the converted value.	
- Failure (Non-ANSI):	Returns null.
- Failure (ANSI Mode):	Returns null instead of failing.





In [0]:
"""
from pyspark.sql.functions import col

# Standard cast (may fail in ANSI mode if data is invalid)
df.withColumn("age", col("age_str").cast("int"))

# Try cast (returns NULL if conversion is impossible)
# Available in PySpark 4.0+
df.withColumn("age", col("age_str").try_cast("int"))

df.selectExpr("try_cast(age_str AS int) as age")


"""

'\nfrom pyspark.sql.functions import col\n\n# Standard cast (may fail in ANSI mode if data is invalid)\ndf.withColumn("age", col("age_str").cast("int"))\n\n# Try cast (returns NULL if conversion is impossible)\n# Available in PySpark 4.0+\ndf.withColumn("age", col("age_str").try_cast("int"))\n\ndf.selectExpr("try_cast(age_str AS int) as age")\n\n\n'


![image_1774641851028.png](./image_1774641851028.png "image_1774641851028.png")

-----------------------------------------------------------
####     7  Dates(datediff, interval)

- Folder: CH06-Working with Data Types
- Notebook: 04-Working with dates

######  Convert String to date        
######  Add, Subtract days and months to date
######  Current date, date difference, and interval
######  Format date 
######  casting to date fails

-----------------------------------------------------------

##### 1 string field containing date values
A string field containing date values often cannot be cast directly in PySpark using a simple.cast("date") because PySpark expects a default, ISO-compliant date format of yyyy-MM-dd. 

If your input string is in any other format (e.g., MM/dd/yyyy, dd-MM-yyyy, or yyyyMMdd), the direct cast will likely result in null values or a SparkDateTimeException error.

#####2 Carefull with columns cotaining STRING 'null'/'Null' values
Columns containing a string 'null'/'Null' values  INSTEAD of containing actual nulls or no values can cause SEVERAL ERRORS

#####3 expr('try_to_date') 
try_to_date' is a string function, so you do not need to import it, that tries to cast into date, if it fails to do so it will return a Null value.

#####4 date_format 
when date_format is used it will return  a STRING not a date.




![image_1774890256343.png](./image_1774890256343.png "image_1774890256343.png")

![image_1774890573091.png](./image_1774890573091.png "image_1774890573091.png")

#### datediff()

The pyspark.sql.functions import datediff() function in PySpark calculates the number of days between two dates by evaluating end_date - start_date

Parameters:
1)      end: The end date column (Minuend).
2)      start: The start date column (Subtrahend).

3)      If end is after start, the result is positive.
4)      If end is before start, the result is negative.



#### TILL HERE